# Практика · ML Handbook гл. 43 (43 representation learning)

Источник: [representation-learning.html](https://mlhandbook-production.up.railway.app/parts/deep-learning/representation-learning.html)

Теория: [../d.5.43 Обучение представлений и metric learning.md](../d.5.43 Обучение представлений и metric learning.md) · [README.md](./README.md)

**Handbook-код — в этом файле**. Эталон: [handbook_gl43_практика_решения.ipynb](./handbook_gl43_практика_решения.ipynb) **после** попытки.

| ☐ | Блок |\n|---|------|\n| ☐ | §43.1 nn.Embedding |\n| ☐ | §43.2 triplet на digits (укорочено) |


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


---
# §43.1 · Embedding



In [ ]:
emb = nn.Embedding(num_embeddings=1000, embedding_dim=16)
ids = torch.tensor([5, 42, 7])
print('emb', tuple(emb(ids).shape))


---
# §43.2 · triplet (smoke, 50 шагов)



In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
import numpy as np

d = load_digits()
X = torch.tensor((d.data / 16.0).astype('float32'))
y = torch.tensor(d.target)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
encoder = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 16))
opt = torch.optim.Adam(encoder.parameters(), lr=1e-3)
rng = np.random.default_rng(0)

def sample_triplets(y, n=64):
    y_np = y.numpy()
    a, p, n_idx = [], [], []
    for _ in range(n):
        cls = rng.integers(0, 10)
        same = np.where(y_np == cls)[0]
        diff = np.where(y_np != cls)[0]
        a.append(rng.choice(same))
        p.append(rng.choice(same))
        n_idx.append(rng.choice(diff))
    return a, p, n_idx

for step in range(50):
    ai, pi, ni = sample_triplets(ytr)
    za, zp, zn = encoder(Xtr[ai]), encoder(Xtr[pi]), encoder(Xtr[ni])
    loss = F.triplet_margin_loss(za, zp, zn, margin=0.5)
    opt.zero_grad()
    loss.backward()
    opt.step()
print('last loss', round(loss.item(), 4))
